# CNN에서 YOLO로 — 학습·결과 생성 노트북

이 노트북은 **내 저장소의 `hw.py`를 불러와** 학습을 돌리고 `results/` 폴더(결과 파일 3개)를 만듭니다.

1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 를 고르세요.
2. 아래 첫 셀의 `REPO_URL` 을 내 저장소 주소로 바꾸세요.
3. **런타임 → 모두 실행**. 20~30분 걸립니다.
4. 마지막 셀이 내려받게 해 주는 `results.zip` 을 풀어 GitHub 저장소의 `results/` 에 올리세요 (README 의 "제출 방법").

`hw.py` 를 고쳤다면 GitHub 에 먼저 커밋하고, 이 노트북을 **처음부터 다시** 실행해야 합니다.

In [ ]:
# ── 설정 ─────────────────────────────────────────────
REPO_URL = "https://github.com/내아이디/hw-cnn-to-yolo"   # ← 내 저장소 주소로 바꾸세요
CNN_EPOCHS = 5          # SmallCNN 학습 에폭 (바꿔도 됨)
YOLO_EPOCHS = 3         # YOLO 분류 학습 에폭 (바꿔도 됨)
YOLO_IMGSZ = 32         # YOLO 분류 입력 크기 (32의 배수)
MY_PHOTO = None         # 사진 경로를 직접 써도 됨 (비워 두면 실행 중에 올리기 버튼이 나옴)
QUICK = False           # True: 일부 데이터로 빠르게 점검만 (제출용 아님)

In [ ]:
# ── 준비: 저장소 내려받기, 패키지 설치 ─────────────────
import os, sys, subprocess, importlib
if not os.path.exists('hw.py'):                       # 이미 저장소 안이면 건너뜀
    name = REPO_URL.rstrip('/').split('/')[-1]
    if not os.path.exists(name):
        subprocess.run(['git', 'clone', '-q', REPO_URL, name], check=True)
    os.chdir(name)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
sys.path.insert(0, os.getcwd())
os.makedirs('results', exist_ok=True)
import torch, hw
importlib.reload(hw)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
ENV = torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'
print('저장소:', os.getcwd(), '| 장치:', ENV)

## Part 1. CNN 기초 — 내 `hw.py` 함수 확인

In [ ]:
for n, k, s, p in [(28, 3, 1, 1), (28, 3, 2, 1), (28, 5, 1, 0), (7, 2, 2, 0)]:
    print(f'conv_out({n}, {k}, {s}, {p}) = {hw.conv_out(n, k, s, p)}')
cnn = hw.SmallCNN()
print('SmallCNN 파라미터:', f'{hw.count_params(cnn):,}')
print('출력 모양:', tuple(cnn(torch.zeros(2, 1, 28, 28)).shape))

In [ ]:
# ── Fashion-MNIST 로 SmallCNN 학습 ─────────────────────
import time
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

torch.manual_seed(0)
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_set = datasets.FashionMNIST('data', train=True, download=True, transform=tf)
test_set = datasets.FashionMNIST('data', train=False, download=True, transform=tf)
if QUICK:
    train_set, test_set = Subset(train_set, range(3000)), Subset(test_set, range(1000))
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader = DataLoader(test_set, batch_size=512)

cnn = hw.SmallCNN().to(device)
opt = torch.optim.Adam(cnn.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()
epochs = 1 if QUICK else CNN_EPOCHS
t0 = time.time()
for ep in range(epochs):
    cnn.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(); loss = loss_fn(cnn(x), y); loss.backward(); opt.step()
    print(f'epoch {ep + 1}: loss {loss.item():.3f}  test acc {hw.evaluate(cnn, test_loader, device):.4f}')
cnn_seconds = time.time() - t0
cnn_acc = hw.evaluate(cnn, test_loader, device)
print(f'SmallCNN 최종 시험 정확도 {cnn_acc:.4f}  ({cnn_seconds:.0f}초)')

In [ ]:
# ── 시험 이미지 예측 그림 → results/predictions.png ───
import matplotlib.pyplot as plt
CLASSES = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot']
x, y = next(iter(test_loader))
cnn.eval()
with torch.no_grad():
    pred = cnn(x[:16].to(device)).argmax(1).cpu()
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(x[i, 0], cmap='gray')
    ax.set_title(f'{CLASSES[pred[i]]}\n(true {CLASSES[y[i]]})', fontsize=8, color='green' if pred[i] == y[i] else 'red')
    ax.axis('off')
plt.tight_layout(); plt.savefig('results/predictions.png', dpi=100); plt.show()

## Part 2. YOLO 안의 CNN

YOLOv8n 을 불러와 **내 `yolo_grid_sizes`, `yolo_num_predictions`, `find_stride2_convs`** 가 실제 모델과 맞는지 확인합니다.

In [ ]:
from ultralytics import YOLO
det = YOLO('yolov8n.pt')
net = det.model.eval()
for p in net.parameters():          # ultralytics 는 추론용으로 불러오며 requires_grad 를 꺼 둔다
    p.requires_grad = True
print('YOLOv8n 파라미터:', f'{hw.count_params(net):,}')

seen = {}
h = net.model[-1].register_forward_hook(lambda m, inp, out: seen.__setitem__('x', inp[0]))
for s in [640, 320]:
    with torch.no_grad():
        out = net(torch.zeros(1, 3, s, s))
    out = out[0] if isinstance(out, (list, tuple)) else out
    real = [t.shape[-1] for t in seen['x']]
    print(f'{s}: 실제 격자 {real} / 내 계산 {hw.yolo_grid_sizes(s)}  |  실제 출력 {tuple(out.shape)} / 내 계산 {hw.yolo_num_predictions(s)}')
h.remove()

convs = hw.find_stride2_convs(net)
print(f'stride 2 conv {len(convs)}개:')
for c in convs:
    print('  ', c)

In [ ]:
# ── 내 사진으로 탐지 → results/detection.png ──────────
# 직접 찍은 사진(강의실, 거리, 책상 등)을 올리세요. 올리지 않으면 예시 사진(bus.jpg)을 씁니다 — 제출용으로는 내 사진을 쓰세요.
img_path = MY_PHOTO
try:
    if img_path is None:
        from google.colab import files
        up = files.upload()
        if up:
            img_path = list(up)[0]
except Exception:
    pass
if img_path is None:
    from ultralytics.utils import ASSETS
    img_path = str(ASSETS / 'bus.jpg')
r = det.predict(img_path, conf=0.25, verbose=False)[0]
r.save('results/detection.png')
detections = [{'name': det.names[int(b.cls)], 'conf': round(float(b.conf), 3),
               'box': [round(v, 1) for v in b.xyxy[0].tolist()]} for b in r.boxes]
for d in detections:
    print(d)

## Part 3. 같은 데이터로 YOLO 분류 모델 학습

In [ ]:
cls = YOLO('yolov8n-cls.pt')
t0 = time.time()
cls.train(data='fashion-mnist', epochs=1 if QUICK else YOLO_EPOCHS, imgsz=YOLO_IMGSZ, batch=128,
          fraction=0.05 if QUICK else 1.0, plots=False, verbose=False)
yolo_seconds = time.time() - t0
m = cls.val(data='fashion-mnist', imgsz=YOLO_IMGSZ, plots=False, verbose=False)
yolo_params = sum(p.numel() for p in cls.model.parameters())
print(f'YOLOv8n-cls top1 {m.top1:.4f}  파라미터 {yolo_params:,}  ({yolo_seconds:.0f}초)')

In [ ]:
# ── results.json 저장, results.zip 내려받기 ─────────────
import json, shutil
res = {
    'environment': ENV + (' (QUICK)' if QUICK else ''),
    'smallcnn': {'params': hw.count_params(hw.SmallCNN()), 'epochs': epochs,
                 'test_acc': round(cnn_acc, 4), 'train_seconds': round(cnn_seconds, 1)},
    'yolo_cls': {'model': 'yolov8n-cls.pt', 'params': yolo_params, 'epochs': 1 if QUICK else YOLO_EPOCHS,
                 'imgsz': YOLO_IMGSZ, 'top1': round(float(m.top1), 4), 'train_seconds': round(yolo_seconds, 1)},
    'yolo_det': {'image': os.path.basename(img_path), 'detections': detections},
}
with open('results/results.json', 'w', encoding='utf-8') as f:
    json.dump(res, f, ensure_ascii=False, indent=2)
print(json.dumps(res, ensure_ascii=False, indent=2)[:1500])
shutil.make_archive('results', 'zip', 'results')
try:
    from google.colab import files
    files.download('results.zip')
except Exception:
    print('results.zip 저장됨:', os.path.abspath('results.zip'))